In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

In [0]:
#initializing session
spark= SparkSession.builder.appName("QA").getOrCreate()

In [0]:
base_dir= "/FileStore/tables/"

#loading raw csv
Fact_Transaction_raw = spark.read\
    .format("csv")\
        .option("header", True)\
            .option("inferSchema",True)\
                .load(base_dir+"Fact_Transactions.csv")
#loading delta file
Fact_Transaction_Delta = spark.read\
    .format("delta")\
        .load(base_dir + "Fact_Transaction")

In [0]:
Fact_Transaction_Delta = Fact_Transaction_Delta.withColumnRenamed("Amount", "factAmount")
Fact_Transaction_raw = Fact_Transaction_raw.withColumnRenamed("Amount", "rawAmount")


Performing full outer join 

In [0]:

diff_df = Fact_Transaction_raw.alias("raw")\
    .join(Fact_Transaction_Delta.alias("final"),
          on="TransactionID",
          how="fullouter")


In [0]:
diff_df.display()

TransactionID,ProductCategory,StoreRegion,CustomerType,rawAmount,ProductCategoryID,CustomerTypeID,RegionID,MappingLabelId,factAmount
1,Laptop,North,Retail,1200,101,1,1,null,1200
2,Mobile,South,Wholesale,800,102,2,2,null,800
3,Tablet,East,Retail,300,103,1,3,c799a87abeca2fd57c6289af216b8b08d55339b16291995887c7ec2096ffe5aa,300
4,Laptop,West,Retail,1500,101,1,4,eefb8f08341fe3b9efcd42a96738ac9e0dd58c31ec5501fa1dabf59ea7332fe8,1500
5,Tablet,North,Wholesale,400,103,2,1,null,400
6,Mobile,East,Retail,900,102,1,3,null,900
7,Mobile,South,Retail,850,102,1,2,null,850
8,Laptop,West,Wholesale,1700,101,2,4,null,1700
9,Tablet,East,Retail,350,103,1,3,c799a87abeca2fd57c6289af216b8b08d55339b16291995887c7ec2096ffe5aa,350
10,Mobile,North,Retail,950,102,1,1,null,950


In [0]:
diff_df = diff_df.select("TransactionID", "rawAmount", "factAmount")

In [0]:
diff_df_difference = diff_df.withColumn("DifferenceInAmount", col("rawAmount")- col("factAmount"))

In [0]:
diff_df_difference.display()

TransactionID,rawAmount,factAmount,DifferenceInAmount
1,1200,1200,0
2,800,800,0
3,300,300,0
4,1500,1500,0
5,400,400,0
6,900,900,0
7,850,850,0
8,1700,1700,0
9,350,350,0
10,950,950,0


In [0]:
#select if anycolumn has difffernce not equal to 0
diff_df_difference.filter(col("DifferenceInAmount") != 0).display()

TransactionID,rawAmount,factAmount,DifferenceInAmount


Since, 0 row were selected, there is no data loss